# H1: Comprehensive Project Planning Template

Use this template to prepare your H1 problem definition document. H1 is a plan for the H2 implementation, so your goal is to define a realistic, testable project before you start building.

**Target length:** approximately 1200-1800 words total, excluding tables.

## Student Details

- **Muhammad Wildan**
- **Student ID:36065315**
- **Project title: Basketball Shot Quality Classifier for Game Review**
- **Tutor: Chu Xian Loong**

## H1.1 Problem Definition

### Problem

Describe the problem in your own words. Be specific about what is happening, why it matters, and what part of the problem your project will address.

Coaches and players are biased in their analysis of game film in a "results-oriented" way, such as when a player takes a 20-foot shot off of the block and makes it, it's usually celebrated while a player misses a wide open catch and shoot 3-point shot is usually frowned upon. Each game, 80 to 100 shot attempts need to be manually reviewed to differentiate shot decision quality (good process) from shot outcome (make or miss) which takes many hours of coaching time. 

This project does not seek to track every player's movement as is done in the wider field of sports analysis with multi-camera computer vision, but rather it aims to have an AI model score and classify the shot attempt as either a "High-Quality Shot" or "Low-Quality (Forced) Shot" based on five basic contextual measurements of the shot attempt (distance to the basket, distance to closest defender, time left on the shot clock, number of dribbles taken, and touch time). The coach can easily use this slice to filter post-game shot logs and identify right-offensive habits to correct.

### Stakeholders And Users

Who experiences this problem? Who would use your system? Who could be affected by its outputs or decisions?

Primary User: The amateur or university basketball coach (or team analytics secretary) with limited time for film review (30-45 minutes per week), who has to determine what offensive sets to implement and what bad shot selections to work on in practice.
Affected Stakeholders: Point guards, wings and forwards who are affected by their playing time, role, and coaching feedback as a result of how their shot selection is graded.
Secondary Users: Student club analysts, who record and summarise game statistics for the coaching staff.
### Why AI Is Appropriate

A manual review process is too slow and subjective, while a fixed rule-based system (for example, IF shot_distance < 10 feet THEN Good Shot ELSE Bad Shot) fails because basketball variables interact continuously and unpredictably. A 6-foot layup contested by a defender 1 foot away after 8 dribbles with 2 seconds left on the shot clock is a much worse decision than a wide-open 23-foot three-pointer with 0 dribbles and 14 seconds on the clock, even though in a ceteris paribus setting it is the opposite. An AI approach, specifically a linear classifier, whose feature weights are learned from thousands of real shots via gradient descent is very appropriate because it can automatically learn the exact mathematical trade-offs between shooter distance, defender smother, and ball-handling time directly from historical open source data.

### Scope For H2

What will you implement in H2? What is explicitly out of scope? Keep this realistic for approximately 10-15 hours of implementation work.

**In scope:**

- Processing, cleaning, and normalizing a 10,000-shot subset of the public NBA Shot Logs dataset using five numerical features (SHOT_DIST, CLOSE_DEF_DIST, SHOT_CLOCK, DRIBBLES, TOUCH_TIME).
- Implementing two baselines: a Majority-Class predictor and a fixed Distance-Rule classifier.
- Implementing a weighted sum classifier, margin confidence scorer (from P8b), and a gradient descent training looping on squared error loss (from P9).
- Evaluating accuracy, precision, recall, F1-score, and confusion matrices across three court zones (Paint, Mid-Range, and Three-Point).
- Visualizing the training loss curve, learned feature weights, and a 2D decision boundary plot.

**Out of scope:**

- Wrangling and auditing the obtained data.
- Computer vision or extracting tracking data from raw video footage.
- Adjusting predictions for individual NBA player shooting ratings (the model will evaluate shot quality for an "average" shooter).
- Building a user bench dashboard.
- Potentially deep neural networks or complex ensemble models.

### Success Criteria

What would count as a successful H2 outcome? Define success in a way that can be evaluated.

1. The custom gradient descent training loop converges smoothly (training loss decreases without diverging) within 100 iterations.
2. The trained model outperforms both the Majority-Class baseline and the fixed Distance-Rule baseline by at least 3.0 percentage points on Macro F1-score on a held-out test set.
3. The learned weights match physical basketball reality (specifically, SHOT_DIST learns a negative weight and CLOSE_DEF_DIST learns a positive weight) and pass at least 3 of 4 hand-crafted basketball scenario sanity checks.

## H1.2 Risk Register

List at least six specific risks. Include at least one data risk, one technical risk, one ethical or social risk, and one evaluation risk.

| Risk | Category | Likelihood | Impact | Mitigation |
|------|----------|------------|--------|------------|
|Missing Shot-Clock Data:** Roughly 4.3% of rows in the NBA Shot Logs have NaN for SHOT_CLOCK (occurring when the game clock is under 24 seconds at the end of a quarter), crashing np.sum(w*x) | Data | High | Medium | If any row has missing SHOT_CLOCK, replace missing SHOT_CLOCK values with the dataset median (12.2s) and remove around 0.2% of sensor-error rows where TOUCH_TIME<0. | 
|Inherent Outcome Noise in Basketball: Just like the XOR problem in P8b, basketball shots are not 100% linearly separable—wide-open layups sometimes miss (-1), and heavily contested jumpers sometimes go in (+1) | Data | High | High | Use the P8b margin calculations to measure shot confidence/quality rather than expecting 90%+ raw make/miss accuracy, and define success as beating the baseline rule by >= 3% |
| SHOT_DIST goes up to 45 feet while TOUCH_TIME is mostly 0 to 5 seconds. Large feature numbers make the loss bowl very steep, causing gradient descent to overshoot and bounce back and forth (like alpha = 0.95 in P9) | Technical | High | High |Divide each feature column by its maximum value so all 5 features sit between 0 and 1 before running gradient descent. What triggers: If loss increases on any step, divide the learning rate alpha by 5|
| Bad Starting Weights or Local Traps:** As seen in P9 Part 4, starting gradient descent at a poor initial point can slow down convergence or trap the weights| Technical | Low | Medium | Run the gradient descent training loop from 3 different starting weight vectors (all zeros, small random numbers, and hand-guessed weights) and keep the weights with the lowest final training loss |
|Players who get passed the ball with only 1 or 2 seconds left on the shot clock are forced to heave a bad shot. A coach using the system blindly might unfairly punish those players | Ethical| Medium| Medium | Add a simple check in the prediction output that flags any shot with SHOT_CLOCK <= 3.0 seconds as a "Late-Clock Bailout" so the coach reviews the team possession rather than blaming the shooter|
|Misleading Accuracy on Three-Pointers:** Because 3-pointers only go in ~35% of the time compared to ~60% for layups, a classifier could get decent overall accuracy by predicting -1 (Bad Shot) for every 3-pointer |Evaluation | Medium | High |Use P8b's count_confusion, precision, and recall functions separately on Paint (<10ft), Mid-Range (10-22ft), and 3-Point (>22ft) shots. If Recall on wide-open 3-pointers (CLOSE_DEF_DIST > 6ft) is 0%, shift the decision threshold from 0 to a slightly negative margin threshold |

Which risk is most serious, and what will you do if it occurs?

Risk #6 (Misleading Accuracy on Three-Pointers) combined with Risk #2 (Linear Inseparability) is the most serious risk. Because overall misses (-1) make up 54.8% of the dataset and three-point shots miss ~65% of the time, the classifier might learn a huge negative weight for SHOT_DIST and simply label every shot outside 22 feet as -1 (Bad Shot). As we learned in P8b Part 4b, overall accuracy hides this kind of error. If my zone breakdown in Step 4 shows that the model is calling wide-open three-pointers -1, I will use the P8b margin score to lower the cutoff threshold for three-point shots (or weight 3-point makes higher in the loss function to reflect that they are worth 3 points instead of 2) so wide-open perimeter shots are properly recognized as good decisions

## H1.3 Data Sources

### Data Description

What data will you use for training, testing, or experimentation? Describe the features, labels, states, moves, images, text, or measurements it contains.

I will use 5 numerical features: shot distance, closest defender distance, shot clock remaining, dribbles taken, and touch time. The target label is field goals made, which I will convert to +1 for a made shot and -1 for a missed shot, just like we did in the P8b exercises.
### Source, Access, And Licence

Where does the data come from? Include links where relevant. Are there licence, privacy, copyright, consent, or access constraints?

- **Source & Link:** Publicly available on Kaggle ([NBA Shot Logs Dataset](https://www.kaggle.com/datasets/dansbecker/nba-shot-logs)) as a single shot_logs.csv file.
- **Access & Licence:** Public domain (CC0) educational dataset with no paywall. Because it only contains public sports game statistics, there are no privacy or consent issues. I will save the 10,000-row CSV file directly in my project folder so the notebook runs offline.
### Biases, Gaps, And Limits

What might be missing, noisy, biased, too small, or unrepresentative about the data?

Looking at the dataset reveals three main limitations that shape the scope of H2:
1. **Missing Values & Slight Class Imbalance:** 54.8% of the shots are misses (-1) and 45.2% are makes (+1). Also, 4.3% of rows have missing SHOT_CLOCK values (at the end of quarters when the shot clock is turned off), and 0.2% of rows have negative TOUCH_TIME due to tracking glitches. These will be cleaned in Step 1.
2. **NBA Skill Bias:** NBA players make tough shots at a higher rate than university or amateur players. However, the basic geometry of basketball (closer shots and further defenders are better) is the same at every level.
### Fallback Plan

What will you use if the data source is unavailable, unsuitable, or too difficult to process?

If there are any issues loading or cleaning the Kaggle CSV file, my Plan B is to write a simple NumPy function (generate_synthetic_shots(n=5000)) that generates 5,000 synthetic shots with the exact same 5 columns using known basketball rules (for example, shots under 6 feet with a defender > 4 feet away are labeled +1 65% of the time, while shots over 20 feet with a defender < 2 feet away are labeled -1 80% of the time). This ensures I can still run and test my entire P8b/P9 code pipeline.

## H1.4 Proposed Solution

### Technique

I am directly combining what we did in Week 8 and Week 10. For each shot, the classifier computes a weighted sum (w * x + b). If the score is >= 0, it predicts +1 (Good Shot). If < 0, it predicts -1 (Bad Shot). I'll also calculate the signed margin (from P8b Part 4) to measure how far the shot is from the boundary, which acts as a confidence score. To find the best weights, I'll use Gradient Descent on the mean squared error (from P9) to automatically step downhill instead of guessing by hand.

### Justification

This fits the problem perfectly because the weights are highly explainable. A coach can look at the 5 weights and see exactly how much the model values defender space versus shot distance. Also, the margin score lets the coach rank shots from "most confident good shot" to "worst forced shot", which is way more useful than a basic yes/no prediction.

### Alternative Considered And Rejected

I considered two alternatives from the unit and rejected both:
1. Manual Weight Search (P8b Part 5) / Simple Hard-Threshold Perceptron Rule (D3): Trying to guess 5 weights and a bias by hand is too difficult in 5D space. Furthermore, a basic Perceptron rule only updates on misclassified points and never converges when classes overlap (linear inseparability, as we saw in P8b Part 6). Because made and missed basketball shots overlap heavily, a basic Perceptron would bounce around forever, whereas Gradient Descent on a squared-error bowl (P9) smoothly shrinks its step size as the slope flattens near the minimum.
2. Complex Black-Box Models (e.g., Neural Networks): Rejected because they are outside the scope of our unit tasks and hide how the decision is made, meaning a coach cannot see simple feature weights to explain the feedback to players.

### Inputs And Outputs

**Inputs:**

The 5 scaled shot features, a learning rate alpha, and a number of training steps.

**Outputs:**

The learned weights and bias, the +1 or -1 predictions, the margin scores to rank shot quality, and a text flag for coaching feedback (like "Late-Clock Bailout").

### H2 Implementation Steps

- Data Prep (2.5 hours): Load the CSV, handle missing shot clock values, convert labels to +1/-1, and scale all features between 0 and 1 so the gradients don't explode. Split 80% train, 20% test.
- Baselines (2.0 hours): Adapt my P8b helper functions to work on 5D data. Run the majority-class baseline and the fixed distance-rule baseline.
- Gradient Descent (3.5 hours): Write the squared error loss function and gradient descent loop (expanding P9). Run a learning rate sweep to plot the loss curves and pick the best alpha.
- Evaluation (2.5 hours): Test the model using accuracy, confusion matrices, precision, and recall. Break the results down by Paint, Mid-Range, and 3-Point zones.
- Visualisation (2.5 hours): Run the 4 hand-crafted test scenarios, check the sorted margin table to see what the model got confidently wrong, and plot a 2D decision boundary slice.

### Simplifying Assumptions

- Average Shooter: All shots are evaluated as if taken by a league-average shooter so the model grades the *quality of the opportunity* rather than individual player talent.
- Independent Shots: Each shot is treated independently without tracking the game score or player fatigue.
- 2D Contest: Horizontal distance (CLOSE_DEF_DIST) is assumed to be a good enough measure of defensive contest without 3D vertical jump data.

## H1.5 Evaluation Methodology

### Metrics

I'll use the exact metrics from P8b: Accuracy, Confusion Matrix, Precision, and Recall. Accuracy alone hides errors (like predicting -1 for every shot). Precision will tell the coach what fraction of predicted good shots actually went in, and Recall will show what fraction of actual makes the model caught. I'll also use the margin score to evaluate confidence.

### Baseline

Baseline 1: Always predict -1 (Miss). This gets about 55% accuracy but 0% recall.
Baseline 2: A hand-picked 1-feature rule: predict +1 if distance < 10 feet, else -1. This is much simpler than my model because it ignores defense and the clock, and doesn't use gradient descent.

### Test Cases Or Test Split

I'll do an 80/20 train/test split. I'll also hand-trace 4 specific scenarios:
- Wide-open fastbreak layup (should predict +1 with a high positive margin).
- Heavily contested late-clock heave (should predict -1 with a high negative margin and trigger the bailout flag).
- Wide-open catch-and-shoot 3-pointer (should predict +1).
- Contested mid-range after over-dribbling (should predict -1).

### Failure Analysis

I will print the confusion matrix and look at the sorted margin table from P8b Part 4. The system has failed if:
- The gradient descent weights can't even beat the simple 10-foot distance baseline.
- The learned weights are completely backward (e.g. it learns a positive weight for distance, meaning it thinks further shots are easier).
- It has 0% recall on 3-pointers, meaning it just lazily marks all outside shots as bad.

### Success Threshold

It's a success if gradient descent converges within 100 steps, it beats the baselines by at least 3 percentage points in test accuracy/recall, and the weights make logical basketball sense and pass 3 out of 4 of my hand-traced test cases.

## Milestone Discussion Preparation

Prepare short answers to these questions before meeting your tutor.

1. **Why is this problem suitable for an HD comprehensive project?**
   - It solves a real problem for a basketball coach (filtering post-game shot logs to separate good shot decisions from lucky/unlucky makes and misses) where a simple 1-feature rule fails because shooter distance, defender distance, and dribbles all interact. It fits cleanly into 10–15 hours and directly combines our P8b (weighted sums, margins, precision/recall) and P9 (gradient descent) work on a 5-feature real-world dataset.

2. **What is the simplest baseline, and why is your proposed solution more interesting?**
   - The simplest baselines are guessing -1 (Miss) every time (54.8% accuracy, 0% recall) or a 1-feature rule (predict +1 if distance < 10ft). My solution is much more interesting because instead of guessing weights by hand like in P8b Part 5, it uses Gradient Descent (P9) to automatically learn the best weights for all 5 features at once, and uses the Signed Margin (P8b Part 4) to rank how confident each shot decision is.

3. **What data assumption is most likely to fail?**
   - Linear separability. Just like the XOR problem in P8b Part 6, real basketball shots are not linearly separable—a player can take a perfect wide-open shot and still miss (-1), or take a terrible contested shot and get lucky (+1). Because the labels are noisy, no line can get 100% accuracy, which is why using the margin score and comparing against a baseline is so important.

4. **Which risk in your register is most serious, and what will you do if it happens?**
   - The model predicting -1 (Bad Shot) for every 3-pointer because 3-pointers have a lower make percentage (~35%) than layups (~60%). If my court-zone confusion matrix shows 0% recall on wide-open 3-pointers, I will adjust the margin threshold for 3-point shots (or weight 3-point makes higher in the loss function) so wide-open perimeter shots are graded fairly.

5. **How will your evaluation show that the system failed, not just that it succeeded?**
   - First, by checking Precision and Recall from P8b Part 4b—if the model gets 55% accuracy just by predicting -1 almost every time, Recall will be near 0%, exposing the failure. Second, by checking the learned weights w: if SHOT_DIST has a positive weight or CLOSE_DEF_DIST has a negative weight, the math contradicts basketball reality. Third, by checking the sorted margin table (P8b Part 4) to see if the model is "confidently wrong" on obvious shots like wide-open layups.

6. **Who could be harmed or misled if this system were deployed and got things wrong?**
   - Players who get passed the ball at the very end of the 24-second shot clock (<= 3` seconds left) have no choice but to heave a contested shot. If a coach only looked at their average margin score, those players could be unfairly benched. That is why my system adds a "Late-Clock Bailout" flag whenever SHOT_CLOCK <= 3.0.

## AI Use Statement

Declare any AI tools used while preparing H1. Explain what the AI helped with, what you accepted, what you rejected, and how you checked the output.

I used Google Gemini to help brainstorm and structure this H1 planning document:
- **What AI helped with:** Suggesting public datasets that fit a 10–15 hour scope, formatting the 6 risks into the required categories, and organizing the implementation steps.
- **What I accepted:** I chose the basketball shot-selection idea because I play and captain basketball and understand the domain well. I accepted using the Kaggle NBA Shot Logs dataset and breaking down the evaluation by court zones (Paint, Mid-Range, 3-Point).
- **What I rejected:** In its first draft, the AI suggested advanced machine learning terms we have not covered in FIT1061 (such as sigmoid logistic regression, log-loss, and Macro-F1). I rejected those and asked to redesign the solution using only the exact concepts and functions I implemented myself in P8b (weighted sums, signed margin, confusion matrix, precision, recall, linear inseparability) and P9 (gradient descent on squared error and learning-rate sweeps).
- **How I checked the output:** I verified that every step in H1.4 and H1.5 directly matches the code patterns from my P8b and P9 notebooks and that all rubric requirements are met.

## Submission Checklist

- [ ] H1.1 problem definition is specific, scoped, and stakeholder-aware.
- [ ] H1.2 risk register includes at least six specific risks.
- [ ] H1.2 risk register includes data, technical, ethical/social, and evaluation risks.
- [ ] H1.3 data source is accessible, appropriate, and has a fallback plan.
- [ ] H1.4 proposed solution is technically realistic for H2.
- [ ] H1.4 solution is justified against at least one rejected alternative.
- [ ] H1.5 evaluation includes metrics, baseline, test cases, failure analysis, and success threshold.
- [ ] AI use statement is included, if applicable.
- [ ] Document is uploaded to OnTrack and marked **Ready for Feedback**.
- [ ] Milestone discussion is booked with your tutor.

---

*FIT1061 H1: Comprehensive Project Planning Template*